# Continuous Audit — Planner Notifier
Cria um card no bucket STAND-BY por trigger. Dedup pelo próprio Planner:
card ABERTO com o mesmo título é atualizado (descrição, labels e checklist se
vazia), nunca duplicado. Chamadas Graph com retry p/ throttling e validação
de status — falhas aparecem no log. Credenciais nos secrets `planner-*`
(scope `compliance-grc`). Consumido via `%run` após o `utils`.


In [ ]:
import time
import requests
from datetime import datetime
from zoneinfo import ZoneInfo

_BRT   = ZoneInfo("America/Sao_Paulo")
_GRAPH = "https://graph.microsoft.com/v1.0"

# ── Config do Planner (resolvida por NOME a cada rodada — sobrevive a mudanças de ID)
_PLANNER_GROUP_ID  = "b5d2a8c9-e93c-4f88-b09b-0aca8f9c7147"   # grupo GRC
_PLANNER_PLAN_NAME = "Planner_GRC"
_PLANNER_BUCKET    = "STAND-BY"                                # cards novos SEMPRE nascem aqui
_LABEL_HINTS       = ("continuous audit", "gestão de riscos", "dedo no pulso")
_TITLE_PREFIX      = "[Continuous Audit] "

_CHECKLIST = [
    "Analisar os achados no painel e marcar falsos positivos, se houver",
    "Acionar a área responsável e definir o plano de ação",
    "Se for a tratamento: emitir e vincular o apontamento no painel — o alerta fica suprimido até o apontamento fechar",
    "Reavaliar após as ações e confirmar rodada sem achados",
]

_ALERT_LABEL = {"novo_achado": "Novo achado", "reincidente": "Reincidente",
                "persistente": "Achado persistente"}


def _req(method, url, H, json_body=None, expect=(200, 201, 204), tag=""):
    """Chamada Graph com retry para throttling (429) e 5xx; valida o status.
    Retorna o Response ou None (falha após retries — logada, nunca silenciosa)."""
    r = None
    for tentativa in range(4):
        r = requests.request(method, url, headers=H, json=json_body, timeout=30)
        if r.status_code in (429,) or r.status_code >= 500:
            time.sleep(float(r.headers.get("Retry-After", 1 + tentativa)))
            continue
        break
    if r is None or r.status_code not in expect:
        corpo = (r.text[:150] if r is not None else "sem resposta")
        codigo = r.status_code if r is not None else "—"
        print(f"Planner: falha em {tag or method} [{codigo}]: {corpo}")
        return None
    return r


def _graph_headers():
    tenant = dbutils.secrets.get("compliance-grc", "planner-tenant-id")
    cid    = dbutils.secrets.get("compliance-grc", "planner-client-id")
    csec   = dbutils.secrets.get("compliance-grc", "planner-client-secret")
    r = requests.post(
        f"https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token",
        data={"client_id": cid, "client_secret": csec,
              "grant_type": "client_credentials",
              "scope": "https://graph.microsoft.com/.default"},
        timeout=20,
    ).json()
    if "access_token" not in r:
        raise RuntimeError(f"Token Graph falhou: {str(r)[:200]}")
    return {"Authorization": f"Bearer {r['access_token']}",
            "Content-Type": "application/json"}


def _resolve_plan(H):
    plans   = _req("GET", f"{_GRAPH}/groups/{_PLANNER_GROUP_ID}/planner/plans", H,
                   tag="listar planos").json()["value"]
    plan    = next(p for p in plans if p["title"] == _PLANNER_PLAN_NAME)
    buckets = _req("GET", f"{_GRAPH}/planner/plans/{plan['id']}/buckets", H,
                   tag="listar buckets").json()["value"]
    bucket  = next(b for b in buckets
                   if b["name"].strip().upper() == _PLANNER_BUCKET.upper())
    cats    = (_req("GET", f"{_GRAPH}/planner/plans/{plan['id']}/details", H,
                    tag="detalhes do plano").json().get("categoryDescriptions") or {})
    labels  = {k: True for k, v in cats.items()
               if v and any(h in v.lower() for h in _LABEL_HINTS)}
    return plan["id"], bucket["id"], (labels or {"category1": True, "category6": True})


def _open_tasks_by_title(H, plan_id):
    """Dedup direto na fonte: cards ABERTOS do plano (percentComplete < 100) com o
    prefixo do sistema, indexados por título — enxerga inclusive cards manuais."""
    abertos, url = {}, f"{_GRAPH}/planner/plans/{plan_id}/tasks"
    while url:
        r = _req("GET", url, H, tag="listar tasks")
        if r is None:
            break
        body = r.json()
        for t in body.get("value", []):
            if (t.get("percentComplete", 100) < 100
                    and (t.get("title") or "").startswith(_TITLE_PREFIX)):
                abertos.setdefault(t["title"], t)
        url = body.get("@odata.nextLink")
    return abertos


def _description(e, risk, app_url):
    linhas = ["Trigger da Auditoria Contínua — achados que exigem análise.", "",
              f"Teste: {e['test_name']}"]
    if e.get("description"):
        linhas.append(f"O que o teste verifica: {e['description']}")
    if e.get("risco_id") and e["risco_id"] != "N/A":
        extra = ""
        if risk.get("title"):
            extra += f" — {risk['title']}"
        if risk.get("level"):
            extra += f" (Inerente: {risk['level']})"
        linhas.append(f"Risco: {e['risco_id']}{extra}")
    if e.get("area"):
        linhas.append(f"Área responsável: {e['area']}")
    linhas.append(f"Tipo de alerta: {_ALERT_LABEL.get(e['alert'], e['alert'])}")
    linhas.append(f"Achados na rodada: {e['count']}")
    linhas.append(f"Rodada: {datetime.now(_BRT).strftime('%d/%m/%Y %H:%M')} (BRT)")
    if app_url:
        linhas += ["", f"Painel: {app_url}"]
    return "\n".join(linhas)


def _get_details_com_retry(H, task_id):
    """Detalhes de uma task recém-criada podem demorar a propagar — insiste."""
    for _ in range(5):
        r = _req("GET", f"{_GRAPH}/planner/tasks/{task_id}/details", H,
                 expect=(200, 404), tag="detalhes da task")
        if r is not None and r.status_code == 200:
            return r.json()
        time.sleep(1)
    return None


def _patch_details(H, task_id, corpo, tag):
    """PATCH de detalhes com etag fresco e 1 retry em conflito (409/412)."""
    for _ in range(2):
        det = _get_details_com_retry(H, task_id)
        if det is None:
            print(f"Planner: detalhes indisponíveis para {tag} — PATCH não aplicado")
            return False
        r = _req("PATCH", f"{_GRAPH}/planner/tasks/{task_id}/details", 
                 {**H, "If-Match": det["@odata.etag"]}, json_body=corpo,
                 expect=(200, 204, 409, 412), tag=f"atualizar detalhes ({tag})")
        if r is not None and r.status_code in (200, 204):
            return det.get("checklist") is not None and True
        # 409/412 → etag venceu; tenta mais uma vez com etag novo
    print(f"Planner: NÃO consegui gravar checklist/descrição de {tag}")
    return False


def notify_planner_cards(events, risk_info=None, app_url=None,
                         include=("novo_achado", "reincidente")) -> int:
    """Cria/atualiza cards no Planner para os alertas em `include`.

    Dedup pelo próprio Planner: card ABERTO com o título [Continuous Audit]
    {teste} é atualizado (descrição, labels e checklist se estiver vazia) —
    nunca duplicado, mesmo movido de bucket. Card concluído/excluído → novo
    trigger abre card novo. Renomear o título quebra o vínculo.
    Erros de execução não viram card. Opt-out por teste respeitado.
    """
    risk_info = risk_info or {}
    dedup = {}
    for e in events:
        dedup[e["test_name"]] = e
    cards = [e for e in dedup.values() if e["alert"] in include and e["notify"]]
    if not cards:
        print("Planner: nenhum trigger para card.")
        return 0

    H = _graph_headers()
    plan_id, bucket_id, labels = _resolve_plan(H)
    abertos = _open_tasks_by_title(H, plan_id)

    checklist = {str(i): {"@odata.type": "microsoft.graph.plannerChecklistItem",
                          "title": s, "isChecked": False}
                 for i, s in enumerate(_CHECKLIST, 1)}
    criados = atualizados = 0

    for e in cards:
        titulo = f"{_TITLE_PREFIX}{e['test_name']}"
        risk   = risk_info.get(e.get("risco_id"), {})
        desc   = _description(e, risk, app_url)

        existente = abertos.get(titulo)
        if existente:
            # Garante as labels (inclui a tag Continuous Audit) no card existente
            aplicadas = {**(existente.get("appliedCategories") or {}), **labels}
            _req("PATCH", f"{_GRAPH}/planner/tasks/{existente['id']}",
                 {**H, "If-Match": existente["@odata.etag"]},
                 json_body={"appliedCategories": aplicadas},
                 expect=(200, 204, 409, 412), tag=f"labels de {e['test_name']}")
            # Descrição sempre; checklist só se estiver vazia (não sobrescreve progresso)
            det = _get_details_com_retry(H, existente["id"])
            corpo = {"description": desc}
            if det is not None and not (det.get("checklist") or {}):
                corpo["checklist"] = checklist
                corpo["previewType"] = "checklist"
            _patch_details(H, existente["id"], corpo, e["test_name"])
            atualizados += 1
            print(f"Planner ↻ card aberto atualizado: {e['test_name']}")
            time.sleep(0.5)
            continue

        r = _req("POST", f"{_GRAPH}/planner/tasks", H, json_body={
            "planId": plan_id, "bucketId": bucket_id,
            "title": titulo, "priority": 3,
            "appliedCategories": labels,
        }, expect=(200, 201), tag=f"criar card de {e['test_name']}")
        if r is None:
            continue
        task_id = r.json()["id"]
        ok = _patch_details(H, task_id,
                            {"description": desc, "checklist": checklist,
                             "previewType": "checklist"}, e["test_name"])
        criados += 1
        print(f"Planner + card criado: {e['test_name']}" +
              ("" if ok is not False else "  (SEM detalhes — ver avisos acima)"))
        time.sleep(0.5)

    print(f"Planner: {criados} criado(s) · {atualizados} atualizado(s).")
    return criados
